In [1]:
library(progress)
install.packages("dbarts")
install.packages("stochtree")
library(stochtree)
library(dbarts)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘BH’



Attaching package: ‘dbarts’


The following object is masked from ‘package:stochtree’:

    bart




In [8]:
# Set parameters for the simulation
n_replications <- 100
n_sample <- 1500

# This list will store the results of all replications
simulation_results <- vector("list", n_replications)

# --- Main Simulation Loop ---
for (i in 1:n_replications) {
  set.seed(i)
  # Print progress
  if (i %% 20 == 0) {
    cat(sprintf("Running replication: %d / %d\n", i, n_replications))
  }

  # 1. Generate the base covariates (x) for n=500
  # These are common to all 4 scenarios within this replication
  x1 <- rnorm(n_sample)
  x2 <- rnorm(n_sample)
  x3 <- rnorm(n_sample)
  x4 <- rbinom(n_sample, size = 1, prob = 0.5) # Dichotomous (0 or 1)
  x5 <- sample(1:3, size = n_sample, replace = TRUE) # Categorical (1, 2, or 3)

  # Combine covariates into a matrix for convenience
  covariates <- data.frame(x1, x2, x3, x4, x5)

  # 2. Define the helper function g(x)
  # NOTE: As explained, we assume the paper meant g(x5), not g(x4).
  g_values <- c(2, -1, -4) # g(1)=2, g(2)=-1, g(3)=-4
  g_x5 <- g_values[covariates$x5]

  # 3. Define the two prognostic functions (mu)
  mu_linear <- 1 + g_x5 + covariates$x1 * covariates$x3
  mu_nonlinear <- -6 + g_x5 + 6 * abs(covariates$x3 - 1)

  # 4. Define the two treatment effect functions (tau)
  # Homogeneous: constant value
  tau_homogeneous <- rep(3, n_sample)
  # Heterogeneous: depends on x2 and x5
  tau_heterogeneous <- 1 + 2 * covariates$x2 * covariates$x4

  # 5. Generate treatment assignment (T) based on propensity scores (pi)
  # This requires calculating pi for both linear and nonlinear mu

  # Propensity score for the linear mu case
  s_linear <- sd(mu_linear)
  pi_linear <- 0.8 * pnorm(3 * mu_linear / s_linear - 0.5 * covariates$x1) + 0.05 + runif(n_sample) / 10
  # Ensure probabilities are in [0, 1]
  pi_linear <- pmin(1, pmax(0, pi_linear))
  T_linear <- rbinom(n_sample, 1, pi_linear)

  # Propensity score for the nonlinear mu case
  s_nonlinear <- sd(mu_nonlinear)
  pi_nonlinear <- 0.8 * pnorm(3 * mu_nonlinear / s_nonlinear - 0.5 * covariates$x1) + 0.05 + runif(n_sample) / 10
  # Ensure probabilities are in [0, 1]
  pi_nonlinear <- pmin(1, pmax(0, pi_nonlinear))
  T_nonlinear <- rbinom(n_sample, 1, pi_nonlinear)

  # 6. Generate the final observed outcomes (Y) for all 4 scenarios
  # We use a common error term for all scenarios in this replication
  # to reduce Monte Carlo variance when comparing methods.
  epsilon <- rnorm(n_sample, mean = 0, sd = 0.5)

  # Scenario 1: Linear mu, Homogeneous tau
  Y1 <- mu_linear + T_linear * tau_homogeneous + epsilon
  df1 <- data.frame(Y = Y1, T = T_linear, covariates,
                    true_mu = mu_linear, true_tau = tau_homogeneous, true_pi = pi_linear)

  # Scenario 2: Linear mu, Heterogeneous tau
  Y2 <- mu_linear + T_linear * tau_heterogeneous + epsilon
  df2 <- data.frame(Y = Y2, T = T_linear, covariates,
                    true_mu = mu_linear, true_tau = tau_heterogeneous, true_pi = pi_linear)

  # Scenario 3: Nonlinear mu, Homogeneous tau
  Y3 <- mu_nonlinear + T_nonlinear * tau_homogeneous + epsilon
  df3 <- data.frame(Y = Y3, T = T_nonlinear, covariates,
                    true_mu = mu_nonlinear, true_tau = tau_homogeneous, true_pi = pi_nonlinear)

  # Scenario 4: Nonlinear mu, Heterogeneous tau
  Y4 <- mu_nonlinear + T_nonlinear * tau_heterogeneous + epsilon
  df4 <- data.frame(Y = Y4, T = T_nonlinear, covariates,
                    true_mu = mu_nonlinear, true_tau = tau_heterogeneous, true_pi = pi_nonlinear)

  # 7. Store the four generated datasets for this replication
  simulation_results[[i]] <- list(
    linear_homogeneous = df1,
    linear_heterogeneous = df2,
    nonlinear_homogeneous = df3,
    nonlinear_heterogeneous = df4
  )
}

cat("Simulation finished.\n")

# Example: Get the data for the first replication (i=1)
first_replication_data <- simulation_results[[1]]

# Example: Get the "linear, heterogeneous" dataset from the first replication
linear_het_df_rep1 <- first_replication_data$linear_heterogeneous
# Or more directly:
# linear_het_df_rep1 <- simulation_results[[1]]$linear_heterogeneous

# View the first few rows of this dataset
print(head(linear_het_df_rep1))


Running replication: 20 / 100
Running replication: 40 / 100
Running replication: 60 / 100
Running replication: 80 / 100
Running replication: 100 / 100
Simulation finished.
          Y T         x1         x2         x3 x4 x5   true_mu   true_tau
1 -3.507557 0 -0.6264538  0.8500435  0.7391149  1  3 -3.463021  2.7000869
2  3.398106 0  0.1836433 -0.9253130  0.3866087  1  1  3.070998 -0.8506260
3  3.887207 1 -0.8356286  0.8935812  1.2963972  0  1  1.916693  1.0000000
4 -4.106024 0  1.5952808 -0.9410097 -0.8035584  0  3 -4.281901  1.0000000
5  3.456586 1  0.3295078  0.5389521 -1.6026257  0  1  2.471922  1.0000000
6 -3.931586 0 -0.8204684 -0.1819744  0.9332510  1  3 -3.765703  0.6360512
     true_pi
1 0.12793146
2 0.85496837
3 0.93279177
4 0.06968941
5 0.88381490
6 0.07042718


In [9]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
new_colnames <- c(
  # PEHE metrics
  "bcf_1k_pehe1", "bcf_1k_pehe2",
  "bcf_0.5k_pehe1", "bcf_0.5k_pehe2",
  "bcf_0.25k_pehe1", "bcf_0.25k_pehe2",
  "bcf_0.1k_pehe1", "bcf_0.1k_pehe2",
  "bcf_0.05k_pehe1", "bcf_0.05k_pehe2",

  # RMSE metrics (added)
  "bcf_1k_rmse1", "bcf_1k_rmse2",
  "bcf_0.5k_rmse1", "bcf_0.5k_rmse2",
  "bcf_0.25k_rmse1", "bcf_0.25k_rmse2",
  "bcf_0.1k_rmse1", "bcf_0.1k_rmse2",
  "bcf_0.05k_rmse1", "bcf_0.05k_rmse2",

  # MAPE metrics (added)
  "bcf_1k_mape1", "bcf_1k_mape2",
  "bcf_0.5k_mape1", "bcf_0.5k_mape2",
  "bcf_0.25k_mape1", "bcf_0.25k_mape2",
  "bcf_0.1k_mape1", "bcf_0.1k_mape2",
  "bcf_0.05k_mape1", "bcf_0.05k_mape2",

  # Tau 95% interval width metrics
  "bcf_1k_tau_951", "bcf_1k_tau_952",
  "bcf_0.5k_tau_951", "bcf_0.5k_tau_952",
  "bcf_0.25k_tau_951", "bcf_0.25k_tau_952",
  "bcf_0.1k_tau_951", "bcf_0.1k_tau_952",
  "bcf_0.05k_tau_951", "bcf_0.05k_tau_952",

  # Tau 95% interval width metrics (weighted)
  "bcf_1k_tau_951w", "bcf_1k_tau_952w",
  "bcf_0.5k_tau_951w", "bcf_0.5k_tau_952w",
  "bcf_0.25k_tau_951w", "bcf_0.25k_tau_952w",
  "bcf_0.1k_tau_951w", "bcf_0.1k_tau_952w",
  "bcf_0.05k_tau_951w", "bcf_0.05k_tau_952w"
)

# Calculate the total number of columns
num_columns <- length(new_colnames)

# Initialize the results matrix with the correct number of columns
results_matrix <- matrix(NA, nrow = num_simulations, ncol = num_columns)
colnames(results_matrix) <- new_colnames

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

data <- simulation_results[[i]]
linear_het_df <- first_replication_data$linear_heterogeneous
nonlinear_het_df <- first_replication_data$nonlinear_heterogeneous
num_rows <- nrow(linear_het_df)
# Extract the outcome vector Y
Y <- linear_het_df$Y[1:500]

# Extract the treatment vector Z
Z <- linear_het_df$T[1:500]
covariate_names <- grep("^x", names(linear_het_df), value = TRUE)
X <- linear_het_df[1:500 , covariate_names]

# Extract the outcome vector Y
Y_test <- linear_het_df$Y[501:num_rows]

# Extract the treatment vector Z
Z_test <- linear_het_df$T[501:num_rows]

X_test <- linear_het_df[501:num_rows , covariate_names]

p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

Tau_test<- linear_het_df$true_tau[501:num_rows]

num_gfr<-40
n_iter<-25

bcf_1k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_1k_pehe1<-sqrt(mean((Tau_test-rowMeans(bcf_1k$tau_hat_test))^2))

bcf_1k_tau_951<-mean(diag(apply(bcf_1k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_1k_tau_951w<-mean(apply(bcf_1k$tau_hat_test, 1, cred_width, 0.95))

bcf_1k_rmse1 <- sqrt(mean((Y_test - bcf_1k$y_hat_test)^2))
bcf_1k_mape1 <- mean((abs(Y_test - bcf_1k$y_hat_test))/abs(Y))

n_iter<-13

bcf_0.5k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.5k_pehe1<-sqrt(mean((Tau_test-rowMeans(bcf_0.5k$tau_hat_test))^2))

bcf_0.5k_tau_951<-mean(diag(apply(bcf_0.5k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.5k_tau_951w<-mean(apply(bcf_0.5k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.5k_rmse1 <- sqrt(mean((Y_test - bcf_0.5k$y_hat_test)^2))
bcf_0.5k_mape1 <- mean((abs(Y_test - bcf_0.5k$y_hat_test))/abs(Y))

n_iter<-6

bcf_0.25k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.25k_pehe1<-sqrt(mean((Tau_test-rowMeans(bcf_0.25k$tau_hat_test))^2))

bcf_0.25k_tau_951<-mean(diag(apply(bcf_0.25k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.25k_tau_951w<-mean(apply(bcf_0.25k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.25k_rmse1 <- sqrt(mean((Y_test - bcf_0.25k$y_hat_test)^2))
bcf_0.25k_mape1 <- mean((abs(Y_test - bcf_0.25k$y_hat_test))/abs(Y))

n_iter<-3

bcf_0.1k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.1k_pehe1<-sqrt(mean((Tau_test-rowMeans(bcf_0.1k$tau_hat_test))^2))

bcf_0.1k_tau_951<-mean(diag(apply(bcf_0.1k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.1k_tau_951w<-mean(apply(bcf_0.1k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.1k_rmse1 <- sqrt(mean((Y_test - bcf_0.1k$y_hat_test)^2))
bcf_0.1k_mape1 <- mean((abs(Y_test - bcf_0.1k$y_hat_test))/abs(Y))

n_iter<-40

bcf_0.05k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.05k_pehe1<-sqrt(mean((Tau_test-rowMeans(bcf_0.05k$tau_hat_test))^2))

bcf_0.05k_tau_951<-mean(diag(apply(bcf_0.05k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.05k_tau_951w<-mean(apply(bcf_0.05k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.05k_rmse1 <- sqrt(mean((Y_test - bcf_0.05k$y_hat_test)^2))
bcf_0.05k_mape1 <- mean((abs(Y_test - bcf_0.05k$y_hat_test))/abs(Y))


# Extract the outcome vector Y
Y <- nonlinear_het_df$Y[1:500]

# Extract the treatment vector Z
Z <- nonlinear_het_df$T[1:500]
covariate_names <- grep("^x", names(nonlinear_het_df), value = TRUE)
X <- nonlinear_het_df[1:500 , covariate_names]

# Extract the outcome vector Y
Y_test <- nonlinear_het_df$Y[501:num_rows]

# Extract the treatment vector Z
Z_test <- nonlinear_het_df$T[501:num_rows]

X_test <- nonlinear_het_df[501:num_rows , covariate_names]

p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

Tau_test<- nonlinear_het_df$true_tau[501:num_rows]

num_gfr<-40
n_iter<-25

bcf_1k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_1k_pehe2<-sqrt(mean((Tau_test-rowMeans(bcf_1k$tau_hat_test))^2))

bcf_1k_tau_952<-mean(diag(apply(bcf_1k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_1k_tau_952w<-mean(apply(bcf_1k$tau_hat_test, 1, cred_width, 0.95))

bcf_1k_rmse2 <- sqrt(mean((Y_test - bcf_1k$y_hat_test)^2))
bcf_1k_mape2 <- mean((abs(Y_test - bcf_1k$y_hat_test))/abs(Y))

n_iter<-13

bcf_0.5k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.5k_pehe2<-sqrt(mean((Tau_test-rowMeans(bcf_0.5k$tau_hat_test))^2))

bcf_0.5k_tau_952<-mean(diag(apply(bcf_0.5k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.5k_tau_952w<-mean(apply(bcf_0.5k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.5k_rmse2 <- sqrt(mean((Y_test - bcf_0.5k$y_hat_test)^2))
bcf_0.5k_mape2 <- mean((abs(Y_test - bcf_0.5k$y_hat_test))/abs(Y))

n_iter<-6

bcf_0.25k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.25k_pehe2<-sqrt(mean((Tau_test-rowMeans(bcf_0.25k$tau_hat_test))^2))

bcf_0.25k_tau_952<-mean(diag(apply(bcf_0.25k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.25k_tau_952w<-mean(apply(bcf_0.25k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.25k_rmse2 <- sqrt(mean((Y_test - bcf_0.25k$y_hat_test)^2))
bcf_0.25k_mape2 <- mean((abs(Y_test - bcf_0.25k$y_hat_test))/abs(Y))

n_iter<-3

bcf_0.1k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.1k_pehe2<-sqrt(mean((Tau_test-rowMeans(bcf_0.1k$tau_hat_test))^2))

bcf_0.1k_tau_952<-mean(diag(apply(bcf_0.1k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.1k_tau_952w<-mean(apply(bcf_0.1k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.1k_rmse2 <- sqrt(mean((Y_test - bcf_0.1k$y_hat_test)^2))
bcf_0.1k_mape2 <- mean((abs(Y_test - bcf_0.1k$y_hat_test))/abs(Y))

n_iter<-40

bcf_0.05k<-bcf(X, Z, Y, X_test = X_test,
  Z_test = Z_test, propensity_test = p_test,
             propensity_train=p, num_mcmc=n_iter,  num_gfr = num_gfr, general_params = list(num_chains = num_gfr))
bcf_0.05k_pehe2<-sqrt(mean((Tau_test-rowMeans(bcf_0.05k$tau_hat_test))^2))

bcf_0.05k_tau_952<-mean(diag(apply(bcf_0.05k$tau_hat_test, 1, in_cred, Tau_test, 0.95)))

bcf_0.05k_tau_952w<-mean(apply(bcf_0.05k$tau_hat_test, 1, cred_width, 0.95))

bcf_0.05k_rmse2 <- sqrt(mean((Y_test - bcf_0.05k$y_hat_test)^2))
bcf_0.05k_mape2 <- mean((abs(Y_test - bcf_0.05k$y_hat_test))/abs(Y))

# Store the results in the matrix
results_matrix[i, ] <- c(
  # PEHE metrics
  bcf_1k_pehe1, bcf_1k_pehe2,
  bcf_0.5k_pehe1, bcf_0.5k_pehe2,
  bcf_0.25k_pehe1, bcf_0.25k_pehe2,
  bcf_0.1k_pehe1, bcf_0.1k_pehe2,
  bcf_0.05k_pehe1, bcf_0.05k_pehe2,

  # RMSE metrics
  bcf_1k_rmse1, bcf_1k_rmse2,
  bcf_0.5k_rmse1, bcf_0.5k_rmse2,
  bcf_0.25k_rmse1, bcf_0.25k_rmse2,
  bcf_0.1k_rmse1, bcf_0.1k_rmse2,
  bcf_0.05k_rmse1, bcf_0.05k_rmse2,

  # MAPE metrics
  bcf_1k_mape1, bcf_1k_mape2,
  bcf_0.5k_mape1, bcf_0.5k_mape2,
  bcf_0.25k_mape1, bcf_0.25k_mape2,
  bcf_0.1k_mape1, bcf_0.1k_mape2,
  bcf_0.05k_mape1, bcf_0.05k_mape2,

  # Tau 95% interval width metrics
  bcf_1k_tau_951, bcf_1k_tau_952,
  bcf_0.5k_tau_951, bcf_0.5k_tau_952,
  bcf_0.25k_tau_951, bcf_0.25k_tau_952,
  bcf_0.1k_tau_951, bcf_0.1k_tau_952,
  bcf_0.05k_tau_951, bcf_0.05k_tau_952,

  # Tau 95% interval width metrics (weighted)
  bcf_1k_tau_951w, bcf_1k_tau_952w,
  bcf_0.5k_tau_951w, bcf_0.5k_tau_952w,
  bcf_0.25k_tau_951w, bcf_0.25k_tau_952w,
  bcf_0.1k_tau_951w, bcf_0.1k_tau_952w,
  bcf_0.05k_tau_951w, bcf_0.05k_tau_952w
)


}

# Export the results matrix to a CSV file
write.csv(results_matrix, "multiple_wsBCF_simulation_results_DGP1_test.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to multiple_wsBCF_simulation_results_DGP1_test.csv\n")

Simulation completed and results saved to multiple_wsBCF_simulation_results_DGP1_test.csv
